In [1]:
from pathlib import Path

import faiss
import numpy as np
import polars as pl

In [2]:
DATA_DIR = Path("../data")
EMB_PATH = DATA_DIR / "embeddings/item_embeddings.parquet"
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = MODELS_DIR / "faiss_ivfpq.index"
IDS_PATH = MODELS_DIR / "faiss_item_ids.npy"

DIM = 768
NLIST = 16384            # число IVF-ячеек
PQ_M = 64               # число PQ-субвекторов → 64 байта/вектор
FACTORY = f"OPQ{PQ_M},IVF{NLIST},PQ{PQ_M}"
TRAIN_SAMPLE = 1_000_000  # подвыборка для обучения (k-means IVF)
ADD_BATCH = 200_000      # Размер батча для добавления в индекс
NPROBE = 32
SEED = 42
rng = np.random.default_rng(SEED)
print(FACTORY)

OPQ64,IVF16384,PQ64


## Загрузка эмбеддингов всего каталога и L2-нормировка.

In [3]:

items = pl.read_parquet(EMB_PATH)
item_ids = items["item_id"].to_numpy().astype(np.int64)
if "embedding" in items.columns:
    vecs = np.stack(items["embedding"].to_numpy()).astype(np.float32)
else:
    emb_cols = sorted(c for c in items.columns if c.startswith("emb_"))
    vecs = items.select(emb_cols).to_numpy().astype(np.float32)
del items

vecs = np.ascontiguousarray(vecs)
faiss.normalize_L2(vecs)
n, d = vecs.shape
assert d == DIM, f"dim mismatch: {d} != {DIM}"
print(f"Embeddings: {n:,} items  dim={d}")

Embeddings: 2,903,192 items  dim=768


## Cтроим индекс

In [4]:
index = faiss.index_factory(DIM, FACTORY, faiss.METRIC_INNER_PRODUCT)

n_train = min(TRAIN_SAMPLE, n)
train_idx = rng.choice(n, size=n_train, replace=False)
print(f"Training {FACTORY} on {n_train:,} vectors...")
index.train(vecs[train_idx])

for i in range(0, n, ADD_BATCH):
    index.add(vecs[i : i + ADD_BATCH])
    print(f"  added {min(i + ADD_BATCH, n):,}/{n:,}", end="\r")


Training OPQ64,IVF16384,PQ64 on 1,000,000 vectors...


In [5]:
faiss.extract_index_ivf(index).nprobe = NPROBE
print(f"Index built: ntotal={index.ntotal:,}  nprobe={faiss.extract_index_ivf(index).nprobe}")

Index built: ntotal=2,903,192  nprobe=32


## Сохраняем

In [6]:
faiss.write_index(index, str(INDEX_PATH))
np.save(IDS_PATH, item_ids)
size_mb = INDEX_PATH.stat().st_size / 1e6
print(f"Saved → {INDEX_PATH} ({size_mb:.1f} MB)")
print(f"Saved → {IDS_PATH}")

Saved → ..\data\models\faiss_ivfpq.index (262.6 MB)
Saved → ..\data\models\faiss_item_ids.npy


## Проверка

In [8]:
catalog = pl.read_parquet(DATA_DIR / "item_catalog/item_catalog.parquet", columns=["item_id", "title"])
title_by_id = dict(zip(
    catalog["item_id"].to_list(),
    catalog["title"].to_list(),
))

def show_title(iid: int, max_len: int = 60) -> str:
    t = title_by_id.get(int(iid), "—")
    return t if len(t) <= max_len else t[: max_len - 1] + "…"

In [10]:
faiss.extract_index_ivf(index).nprobe = NPROBE
probe = rng.choice(n, size=3, replace=False)
D, I = index.search(vecs[probe], 10)

for row, p in enumerate(probe):
    qid = int(item_ids[p])
    print(f"\nquery item {qid}: {show_title(qid)}")
    for rank, (score, j) in enumerate(zip(D[row], I[row]), start=1):
        if j < 0:
            continue
        nid = int(item_ids[j])
        print(f"  {rank:2d}. {nid}  sim={score:.3f}  {show_title(nid)}")


query item 4326306709: Дом 59 м² на участке 11 сот.
   1. 4326306709  sim=0.973  Дом 59 м² на участке 11 сот.
   2. 7512279459  sim=0.943  Дом 80 м² на участке 40 сот.
   3. 7204093977  sim=0.942  Дом 70 м² на участке 17 сот.
   4. 1890458561  sim=0.941  Дом 68 м² на участке 35 сот.
   5. 7812779571  sim=0.941  Дом 70 м² на участке 45 сот.
   6. 7959157811  sim=0.940  Дом 54 м² на участке 34 сот.
   7. 8065762424  sim=0.940  Дом 70 м² на участке 18 сот.
   8. 7481552491  sim=0.939  Дом 62 м² на участке 20 сот.
   9. 7963408817  sim=0.939  Дом 52 м² на участке 18 сот.
  10. 7500075977  sim=0.938  Дом 67 м² на участке 35 сот.

query item 7756307158: Магнитола
   1. 7756307158  sim=0.963  Магнитола
   2. 8083791917  sim=0.922  Магнитола
   3. 7848622010  sim=0.916  Магнитола
   4. 7809027315  sim=0.915  Магнитола
   5. 8035012443  sim=0.913  Магнитола
   6. 7899483242  sim=0.910  Магнитола 2din
   7. 8035674081  sim=0.909  Магнитола
   8. 8042069692  sim=0.908  Магнитола
   9. 4016940034